In [16]:
"""
统计鉴定结果中的陷阱库肽段
"""
import os

res_dir = "D:/workspace/pFindWorkspace/normal/Gygi_2025_NBT/restrict-new/result/open_dl-0628"
res_path = os.path.join(res_dir, "pFind-Filtered.spectra")

peptides = set()
mod2cnt = dict()
mod2error = dict()  # 存在该类型修饰的肽段中，decoy、陷阱库肽段数目
f = open(res_path,"r")
next(f,None)
for line in f:
    items = line.strip().split("\t")
    seq = items[5]
    mods = items[10]
    peptide  = seq + "\t" + mods
    is_decoy = (items[15]=="decoy")

    proteins = items[12].strip("/").split("/")
    flag = True
    for protein in proteins:
        if not protein.endswith("_ARATH"):
            flag = False
            break

    if peptide not in peptides:
        if mods == "":
            continue
        visit = set()  # 每个肽段出现多个同种修饰，只算作一次
        for pos_mod in mods.strip(";").split(";"):
            _,mod = pos_mod.split(",")
            if mod not in mod2cnt:
                mod2cnt[mod] = 1
            else:
                if mod not in visit:
                    mod2cnt[mod] += 1
            visit.add(mod)
        if flag:
            visit = set()  # 每个肽段出现多个同种修饰，只算作一次
            for pos_mod in mods.strip(";").split(";"):
                _,mod = pos_mod.split(",")
                if mod not in mod2error:
                    mod2error[mod] = 1
                else:
                    if mod not in visit:
                        mod2error[mod] += 1
                visit.add(mod)

    peptides.add(peptide)

In [1]:
"""
统计鉴定结果中的decoy肽段
"""
import os

def get_mod2info(res_path):
    peptides = set()
    mod2cnt = dict()
    mod2error = dict()  # 存在该类型修饰的肽段中，decoy、陷阱库肽段数目

    f = open(res_path,"r")
    next(f,None)
    for line in f:
        items = line.strip().split("\t")
        seq = items[5]
        mods = items[10]
        q_value = float(items[4])
        if q_value>0.01:
            break
        peptide  = seq + "\t" + mods
        is_decoy = (items[15]=="decoy")
    
        flag = False
        if is_decoy:
            flag = True
    
        if peptide not in peptides:
            if mods == "":
                continue
            visit = set()  # 每个肽段出现多个同种修饰，只算作一次
            for pos_mod in mods.strip(";").split(";"):
                _,mod = pos_mod.split(",")
                if mod not in mod2cnt:
                    mod2cnt[mod] = 1
                else:
                    if mod not in visit:
                        mod2cnt[mod] += 1
                visit.add(mod)
            if flag:
                visit = set()  # 每个肽段出现多个同种修饰，只算作一次
                for pos_mod in mods.strip(";").split(";"):
                    _,mod = pos_mod.split(",")
                    if mod not in mod2error:
                        mod2error[mod] = 1
                    else:
                        if mod not in visit:
                            mod2error[mod] += 1
                    visit.add(mod)
    
        peptides.add(peptide)
    f.close()
    return mod2cnt, mod2error
        
res_dir1 = "D:/workspace/pFindWorkspace/normal/Gygi_2025_NBT/restrict-new/result/"
res_path1 = os.path.join(res_dir1, "pFind.spectra")
mod2cnt1, mod2error1 = get_mod2info(res_path1)

res_dir2 = "D:/workspace/pFindWorkspace/normal/Gygi_2025_NBT/restrict-new/result/open_dl-0628/"
res_path2 = os.path.join(res_dir2, "pFind.spectra")
mod2cnt2, mod2error2 = get_mod2info(res_path2)

In [37]:
mod2cnt1

{'Carbamidomethyl[C]': 1}

In [9]:
thr = 100
k=0
m = 0

for mod_name, cnt1 in mod2cnt1.items():
    error_cnt1 = mod2error1[mod_name] if mod_name in mod2error1 else 0
    cnt2 = mod2cnt2[mod_name] if mod_name in mod2cnt2 else 0
    error_cnt2 = mod2error2[mod_name] if mod_name in mod2error2 else 0

    if cnt2 == 0:
        continue
    if (cnt1 >= thr and error_cnt1>0) and (cnt2 >= thr and error_cnt2>0):
        k += 1
        rate1 = error_cnt1/cnt1
        rate2 = error_cnt2/cnt2
        if rate1 > rate2:
            m += 1
        print(mod_name, rate1, rate2)
    # if "Arg-loss" in mod_name:
    #     print(f"'aaaaaaaa{mod_name}':{error_cnt/cnt}"+',')
    # if error_cnt/cnt>0.01:
    #     k += 1
    #     m = max(m, error_cnt/cnt)
    #     print(mod_name, error_cnt, cnt, error_cnt/cnt)
    # if cnt>thr and error_cnt!=0:
    #     k += 1
    #     print(f"'{mod_name}':{error_cnt/cnt}"+',')
print(k, m)

Carbamidomethyl[C] 0.0034329656245708795 0.00294607988672437
Acetyl[ProteinN-term] 0.004536616979909268 0.008214849921011059
Oxidation[M] 0.005550709697882801 0.005630808001260041
Gln->pyro-Glu[AnyN-termQ] 0.005051302288871349 0.005989911727616645
Deamidated[N] 0.013100100177236649 0.014131846306622871
Unknown_302[AnyN-term] 0.0024509803921568627 0.0025078369905956114
Carbamyl[AnyN-term] 0.0015805132019338043 0.0013051179267269508
Phospho[S] 0.0019157088122605363 0.0009505703422053232
AEBS[Y] 0.0006303183107469272 0.0006248047485160887
Unknown_250[AnyN-term] 0.005479452054794521 0.002570694087403599
Formyl[T](Thr->Glu[T]) 0.0018298261665141812 0.0008888888888888889
Val->Thr[V] 0.008547008547008548 0.009009009009009009
Formyl[S](Ser->Asp[S]) 0.0012254901960784314 0.0016266775111834079
Deamidated[Q] 0.0005446623093681918 0.0010845986984815619
Unknown_302[E] 0.004615384615384616 0.002824858757062147
Lys[AnyN-term] 0.001890359168241966 0.0024813895781637717
Val->Pro[V] 0.004065040650406504

In [14]:
a,b = 0,0
for value in mod2cnt1.values():
    a += value
for value in mod2error1.values():
    b += value
print(a,b, b/a)

156495 711 0.004543276143007764


In [28]:
thr = 100
k=0
m = 0

for mod_name, cnt in mod2cnt.items():
    error_cnt = mod2error[mod_name] if mod_name in mod2error else 0
    # if "Arg-loss" in mod_name:
    #     print(f"'aaaaaaaa{mod_name}':{error_cnt/cnt}"+',')
    if error_cnt/cnt>0.01:
        k += 1
        m = max(m, error_cnt/cnt)
        print(mod_name, error_cnt, cnt, error_cnt/cnt)
    # if cnt>thr and error_cnt!=0:
    #     k += 1
    #     print(f"'{mod_name}':{error_cnt/cnt}"+',')
print(k, m)

Deamidated[N] 170 12977 0.013100100177236649
Propionamide_2H(3)[C] 1 30 0.03333333333333333
Arg-loss[AnyC-termR] 35 897 0.03901895206243032
Ala->Thr[A] 2 41 0.04878048780487805
Arg->Npo[R] 1 39 0.02564102564102564
Dehydrated[S] 9 165 0.05454545454545454
Glu->Lys[E] 2 138 0.014492753623188406
Xle->Lys[I] 2 60 0.03333333333333333
Thiophospho[S] 1 71 0.014084507042253521
TMT6plex[K] 2 103 0.019417475728155338
monomethylphosphothione[S] 1 40 0.025
Thr->Ala[T] 1 17 0.058823529411764705
Ser->Val[S] 1 6 0.16666666666666666
Arg->Lys[R] 2 68 0.029411764705882353
Fluoro[A] 1 41 0.024390243902439025
Ala->Ser[A] 6 75 0.08
Xle->Asp[L] 1 73 0.0136986301369863
Ser->Gly[S] 1 21 0.047619047619047616
Dehydrated[T] 7 241 0.029045643153526972
Cation_Fe[III][E] 1 90 0.011111111111111112
monomethylphosphothione[C] 1 80 0.0125
Xle->Thr[I] 1 11 0.09090909090909091
Gly->Ser[G] 1 30 0.03333333333333333
HexNAc(2)[T] 1 24 0.041666666666666664
Lys->Gly[K] 1 39 0.02564102564102564
SPITC[AnyN-term] 1 7 0.14285714285

In [8]:
thr = 100
labels = list()
values = list()
for mod_name,cnt in mod2cnt.items():
    error_cnt = mod2error[mod_name] if mod_name in mod2error else 0
    # print(mod_name, cnt, error_cnt)
    if cnt>=thr and error_cnt!=0:
        labels.append(mod_name)
        values.append(error_cnt/cnt)
        print(f"'{mod_name}':{error_cnt/cnt}"+',')

'Oxidation[M]':0.003167940442719677,
'Carbamidomethyl[C]':0.002061477850565761,
'Gln->pyro-Glu[AnyN-termQ]':0.0006343165239454488,
'Deamidated[N]':0.006276150627615063,
'Unknown_302[AnyN-term]':0.00502828409805154,
'AEBS[Y]':0.0006251953735542357,
'Unknown_250[AnyN-term]':0.010309278350515464,
'Carbamyl[AnyN-term]':0.0019602352282273874,
'Acetyl[ProteinN-term]':0.003504300732717426,
'Formyl[S](Ser->Asp[S])':0.0012219959266802445,
'Dioxidation[W]':0.000574052812858783,
'Cation_Fe[II][D]':0.0013531799729364006,
'Unknown_302[D]':0.0019193857965451055,
'Ammonia-loss[N]':0.002012072434607646,
'Oxidation[W]':0.005208333333333333,
'Carboxymethyl[AnyN-term]':0.011764705882352941,
'Cation_Fe[II][E]':0.0016666666666666668,
'Deamidated[Q]':0.0010857763300760044,
'Unknown_302[E]':0.0028328611898017,
'Carbamyl[K]':0.0004710315591144607,
'Delta_H(6)C(3)O(1)[C]':0.006172839506172839,
'Formyl[T](Thr->Glu[T])':0.0026690391459074734,
'Iodo[Y]':0.003416856492027335,
'Glu->Gln[E]':0.00404040404040404,
'Ca

In [19]:
# names = ["Carboxy[W]","Trp->Kynurenin[W]","Cation_Na[E]","Carbamidomethyl[M]","AEBS[K]","Arg-loss[AnyC-termR]"]
names = ["Arg-loss[AnyC-termR]","Oxidation[W]","Asp->Asn[D]"]
for name in names:
    cnt = mod2cnt[name]
    error_cnt = mod2error[name] if name in mod2error else 0
    print(f"'{name}':{error_cnt/cnt}"+',')

'Arg-loss[AnyC-termR]':0.15384615384615385,
'Oxidation[W]':0.0,
'Asp->Asn[D]':0.0,
